# DHFL with MedSAM Backbone

This notebook ports the DHFL adapter approach from initial `dhfl_segmentation` onto a pretrained 
MedSAM (or SAM ViT-B) image encoder.

## Imports & Device

In [40]:
import copy
import math
import os
import random
from dataclasses import dataclass
from itertools import cycle
from typing import Optional, Tuple
from PIL import Image as PILImage
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor
from torch.utils.data import DataLoader, Dataset

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)
print('torch:', torch.__version__)

device: cpu
torch: 2.5.1+cpu


## Backbone Loader

Tries MedSAM → SAM ViT-B → lightweight ViT fallback in order.
Whichever loads first is used. The rest of the notebook is backbone-agnostic.

In [ ]:
class MedSAMEncoder(nn.Module):
    """
    Wraps the MedSAM / SAM image encoder from HuggingFace.
 
    SAM's patch_embed is a Conv2d fixed at 1024×1024.
    We always resize input to 1024 internally, then the encoder
    outputs [B, 256, 64, 64] (stride-16).

    """
 
    SAM_INPUT_SIZE = 1024
 
    PIXEL_MEAN = [123.675 / 255.0, 116.28 / 255.0, 103.53 / 255.0]
    PIXEL_STD  = [58.395 / 255.0,  57.12 / 255.0,  57.375 / 255.0]
 
    def __init__(self, model_id: str) -> None:
        super().__init__()
        from transformers import SamModel
        print(f'Loading {model_id} ...')
        sam = SamModel.from_pretrained(model_id)
        self.encoder = sam.vision_encoder
        self.feature_channels = 256
        self.model_id = model_id
        mean = torch.tensor(self.PIXEL_MEAN).view(1, 3, 1, 1)
        std  = torch.tensor(self.PIXEL_STD).view(1, 3, 1, 1)
        self.register_buffer('pixel_mean', mean)
        self.register_buffer('pixel_std',  std)
        n = sum(p.numel() for p in self.encoder.parameters())
        print(f'Loaded: {model_id}  ({n/1e6:.1f}M params)')
 
    @property
    def input_size(self) -> int:
        return self.SAM_INPUT_SIZE
 
    def preprocess(self, x: Tensor) -> Tensor:
        """Grayscale→RGB, resize to 1024×1024, normalize."""
        if x.shape[1] == 1:
            x = x.repeat(1, 3, 1, 1)
        if x.shape[-2:] != (self.SAM_INPUT_SIZE, self.SAM_INPUT_SIZE):
            x = F.interpolate(
                x, size=(self.SAM_INPUT_SIZE, self.SAM_INPUT_SIZE),
                mode='bilinear', align_corners=False,
            )
        return (x - self.pixel_mean) / self.pixel_std
 
    def forward(self, x: Tensor) -> Tensor:
        x = self.preprocess(x)
        out = self.encoder(pixel_values=x)
        h = out.last_hidden_state
        if h.ndim == 3:
            B, N, C = h.shape
            grid = int(math.isqrt(N))
            h = h.permute(0, 2, 1).reshape(B, C, grid, grid)
        return h  # [B, 256, 64, 64]
 
 
class FallbackViTEncoder(nn.Module):
    """
    Lightweight ViT fallback if MedSAM / SAM cannot be loaded.
    Output projected to 256 channels to match MedSAM interface.
    """
    def __init__(self, model_id: str = 'WinKawaks/vit-small-patch16-224'):
        super().__init__()
        from transformers import ViTModel
        print(f'Loading fallback ViT: {model_id} ...')
        vit = ViTModel.from_pretrained(model_id)
        self.encoder = vit
        hidden = vit.config.hidden_size
        self.proj = nn.Conv2d(hidden, 256, kernel_size=1)
        self.feature_channels = 256
        self.model_id = model_id
        print(f'Loaded fallback ViT: {model_id}')
 
    @property
    def input_size(self) -> int:
        return 224
 
    def forward(self, x: Tensor) -> Tensor:
        if x.shape[1] == 1:
            x = x.repeat(1, 3, 1, 1)
        if x.shape[-1] != self.input_size:
            x = F.interpolate(x, size=(self.input_size, self.input_size),
                              mode='bilinear', align_corners=False)
        out = self.encoder(pixel_values=x)
        h = out.last_hidden_state[:, 1:]  # drop [CLS]
        B, N, C = h.shape
        grid = int(math.isqrt(N))
        h = h.permute(0, 2, 1).reshape(B, C, grid, grid)
        return self.proj(h)
 
 
def load_backbone(device: torch.device) -> nn.Module:
    attempts = [
        ('medsam',    lambda: MedSAMEncoder('flaviagiammarino/medsam-vit-base')),
        ('sam-vit-b', lambda: MedSAMEncoder('facebook/sam-vit-base')),
        ('vit-small', lambda: FallbackViTEncoder('WinKawaks/vit-small-patch16-224')),
    ]
    for name, factory in attempts:
        try:
            backbone = factory().to(device)
            backbone.eval()
            for p in backbone.encoder.parameters():
                p.requires_grad_(False)
            print(f'\nUsing backbone: {name}')
            return backbone
        except Exception as e:
            print(f'[{name}] failed: {e}')
    raise RuntimeError('All backbone loading attempts failed.')
 
 
BACKBONE = load_backbone(DEVICE)
print(f'Feature channels: {BACKBONE.feature_channels}')
print(f'SAM encoder input: {BACKBONE.input_size}×{BACKBONE.input_size}')
print(f'SAM encoder output: [B, 256, 64, 64]  (your dataset images can stay at 256×256)')


def load_backbone(device: torch.device) -> nn.Module:
    attempts = [
        ('medsam',   lambda: MedSAMEncoder('flaviagiammarino/medsam-vit-base')),
        ('sam-vit-b',lambda: MedSAMEncoder('facebook/sam-vit-base')),
        ('vit-small',lambda: FallbackViTEncoder('WinKawaks/vit-small-patch16-224')),
    ]

    for name, factory in attempts:
        try:
            backbone = factory().to(device)
            backbone.eval()
            # Freeze all encoder weights
            for p in backbone.encoder.parameters():
                p.requires_grad_(False)
            print(f'\nUsing backbone: {name}')
            return backbone
        except Exception as e:
            print(f'[{name}] failed: {e}')

    raise RuntimeError(
        'All backbone loading attempts failed. '
        'Check your internet connection and that transformers is installed.'
    )


BACKBONE = load_backbone(DEVICE)
print(f'Feature channels: {BACKBONE.feature_channels}')
print(f'Expected input size: {BACKBONE.input_size}x{BACKBONE.input_size}')

Loading flaviagiammarino/medsam-vit-base ...


[medsam] failed: Due to a serious vulnerability issue in `torch.load`, even with `weights_only=True`, we now require users to upgrade torch to at least v2.6 in order to use the function. This version restriction does not apply when loading files with safetensors.
See the vulnerability report here https://nvd.nist.gov/vuln/detail/CVE-2025-32434
Loading facebook/sam-vit-base ...


Loading weights:   0%|          | 0/314 [00:00<?, ?it/s]

Loaded: facebook/sam-vit-base  (89.7M params)

Using backbone: sam-vit-b
Feature channels: 256
SAM encoder input: 1024×1024
SAM encoder output: [B, 256, 64, 64]  (your dataset images can stay at 256×256)
Loading flaviagiammarino/medsam-vit-base ...
[medsam] failed: Due to a serious vulnerability issue in `torch.load`, even with `weights_only=True`, we now require users to upgrade torch to at least v2.6 in order to use the function. This version restriction does not apply when loading files with safetensors.
See the vulnerability report here https://nvd.nist.gov/vuln/detail/CVE-2025-32434
Loading facebook/sam-vit-base ...


Loading weights:   0%|          | 0/314 [00:00<?, ?it/s]

Loaded: facebook/sam-vit-base  (89.7M params)

Using backbone: sam-vit-b
Feature channels: 256
Expected input size: 1024x1024


## Model Components

In [ ]:
def _group_count(channels: int) -> int:
    for g in (32, 16, 8, 4, 2, 1):
        if channels % g == 0:
            return g
    return 1
 
 
class ConvBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, stride: int = 1) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False),
            nn.GroupNorm(_group_count(out_ch), out_ch),
            nn.GELU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.GroupNorm(_group_count(out_ch), out_ch),
            nn.GELU(),
        )
    def forward(self, x: Tensor) -> Tensor:
        return self.net(x)
 
 
class ErrorAwareResidualAdapter(nn.Module):
    def __init__(
        self,
        channels: int = 256,
        error_channels: int = 1,
        error_embed_channels: int = 16,
        bottleneck_channels: int = 64,
        zero_init: bool = True,
    ) -> None:
        super().__init__()
        self.error_embed_channels = error_embed_channels
        self.feature_norm = nn.GroupNorm(_group_count(channels), channels)
        self.error_dropout = nn.Dropout(p=0.3)
        self.error_encoder = nn.Sequential(
            nn.Conv2d(error_channels, error_embed_channels, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(error_embed_channels, error_embed_channels, 3, padding=1),
            nn.GELU(),
        )
        self.delta_net = nn.Sequential(
            nn.Conv2d(channels + error_embed_channels, bottleneck_channels, 1),
            nn.GELU(),
            nn.Conv2d(bottleneck_channels, bottleneck_channels, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(bottleneck_channels, channels, 1),
        )
        if zero_init:
            nn.init.zeros_(self.delta_net[-1].weight)
            nn.init.zeros_(self.delta_net[-1].bias)
 
    def _encode_error(self, features: Tensor, error_map: Optional[Tensor]) -> Tensor:
        B, _, H, W = features.shape
        if error_map is None:
            return features.new_zeros(B, self.error_embed_channels, H, W)
        resized = F.interpolate(error_map, size=(H, W), mode='bilinear', align_corners=False)
        return self.error_dropout(self.error_encoder(resized))
 
    def forward(self, features: Tensor, error_map: Optional[Tensor] = None):
        error_embed = self._encode_error(features, error_map)
        delta = self.delta_net(torch.cat([self.feature_norm(features), error_embed], dim=1))
        if error_map is None:
            gate = 1.0
        else:
            error_strength = error_map.abs().mean(dim=(1, 2, 3), keepdim=True)
            gate = 1.0 - torch.exp(-5.0 * error_strength)
        delta = gate * delta
        return features + delta, delta
 
 
# ── Decoder A: Lightweight (~588K params) 

class LightweightDecoder(nn.Module):
    """
    4× bilinear upsample: [B,256,64,64] → [B,1,1024,1024]
    ~588K params. Added Dropout2d for regularization.
    """
    def __init__(self, in_channels: int = 256, out_channels: int = 1,
                 dropout: float = 0.1) -> None:
        super().__init__()
        self.net = nn.Sequential(
            ConvBlock(in_channels, 128),
            nn.Dropout2d(p=dropout),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            ConvBlock(128, 64),
            nn.Dropout2d(p=dropout),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            ConvBlock(64, 32),
            nn.Dropout2d(p=dropout),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            ConvBlock(32, 16),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            nn.Conv2d(16, out_channels, 1),
        )
    def forward(self, x: Tensor) -> Tensor:
        return self.net(x)
 
 
# ── Decoder B: Heavy (~4.5M params) 
 
class ResidualDecoderBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int) -> None:
        super().__init__()
        self.conv = ConvBlock(in_ch, out_ch)
        self.skip = nn.Conv2d(in_ch, out_ch, 1, bias=False) if in_ch != out_ch else nn.Identity()
    def forward(self, x: Tensor) -> Tensor:
        return self.conv(x) + self.skip(x)
 
 
class HeavyDecoder(nn.Module):
    """FPN-style residual decoder: [B,256,64,64] → [B,1,1024,1024]"""
    def __init__(self, in_channels: int = 256, out_channels: int = 1) -> None:
        super().__init__()
        self.s1 = ResidualDecoderBlock(in_channels, 256)
        self.up1 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        self.s2 = ResidualDecoderBlock(256, 128)
        self.up2 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        self.s3 = ResidualDecoderBlock(128, 64)
        self.up3 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        self.s4 = ResidualDecoderBlock(64, 32)
        self.up4 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        self.head = nn.Sequential(
            nn.Conv2d(32, 16, 3, padding=1, bias=False),
            nn.GroupNorm(1, 16),
            nn.GELU(),
            nn.Conv2d(16, out_channels, 1),
        )
    def forward(self, x: Tensor) -> Tensor:
        x = self.up1(self.s1(x))
        x = self.up2(self.s2(x))
        x = self.up3(self.s3(x))
        x = self.up4(self.s4(x))
        return self.head(x)
 
 
@dataclass(frozen=True)
class ForwardDebug:
    logits: Tensor
    features: Tensor
    refined_features: Tensor
    delta_features: Tensor
 
 
class DHFLMedSAM(nn.Module):
    """
    DHFL segmenter with pretrained MedSAM / SAM encoder.
 
    decoder_type:
      'lightweight'  — 588K params, fast, CPU-friendly
      'heavy'        — 4.5M params, FPN-style residual, better for real data
    """
 
    def __init__(
        self,
        backbone: nn.Module,
        adapter_bottleneck: int = 64,
        error_embed_channels: int = 16,
        decoder_type: str = 'lightweight',
    ) -> None:
        super().__init__()
        self.backbone = backbone
        C = backbone.feature_channels
 
        self.adapter = ErrorAwareResidualAdapter(
            channels=C,
            error_embed_channels=error_embed_channels,
            bottleneck_channels=adapter_bottleneck,
        )
        self.decoder = HeavyDecoder(C) if decoder_type == 'heavy' else LightweightDecoder(C)
        self.decoder_type = decoder_type
 
        n_adapt  = sum(p.numel() for p in self.adapter.parameters())
        n_decode = sum(p.numel() for p in self.decoder.parameters())
        print(f'DHFLMedSAM | decoder={decoder_type} | '
              f'adapter={n_adapt/1e3:.1f}K  decoder={n_decode/1e3:.1f}K')
 
    def freeze_for_feedback(self) -> None:
        """Freeze backbone + decoder. Only adapter trains."""
        for p in self.backbone.parameters():
            p.requires_grad_(False)
        for p in self.decoder.parameters():
            p.requires_grad_(False)
        for p in self.adapter.parameters():
            p.requires_grad_(True)
 
    def unfreeze_decoder(self) -> None:
        """Unfreeze decoder for base training (backbone always stays frozen)."""
        for p in self.decoder.parameters():
            p.requires_grad_(True)
 
    def adapter_parameters(self):
        return [p for p in self.adapter.parameters() if p.requires_grad]
 
    def forward(
        self,
        x: Tensor,
        error_map: Optional[Tensor] = None,
        return_debug: bool = False,
    ):
        input_size = x.shape[-2:]  
 
        # Backbone always frozen — no grad needed
        with torch.no_grad():
            features = self.backbone(x)   # [B, 256, 64, 64]
 
        refined, delta = self.adapter(features, error_map)
        logits = self.decoder(refined)    # [B, 1, 1024, 1024]
 
        if logits.shape[-2:] != input_size:
            logits = F.interpolate(logits, size=input_size,
                                   mode='bilinear', align_corners=False)
 
        if return_debug:
            return ForwardDebug(logits, features, refined, delta)
        return logits

## Losses, Metrics, and Feedback Utilities

In [9]:
def _check_binary_tensors(logits: Tensor, targets: Tensor) -> None:
    if logits.shape != targets.shape:
        raise ValueError(f'shape mismatch: {tuple(logits.shape)} vs {tuple(targets.shape)}')
    if logits.ndim != 4 or logits.shape[1] != 1:
        raise ValueError(f'expected [B, 1, H, W], got {tuple(logits.shape)}')


def soft_dice_loss(logits: Tensor, targets: Tensor, eps: float = 1e-6) -> Tensor:
    _check_binary_tensors(logits, targets)
    probs = torch.sigmoid(logits)
    dims = (1, 2, 3)
    inter = torch.sum(probs * targets, dim=dims)
    denom = torch.sum(probs, dim=dims) + torch.sum(targets, dim=dims)
    return (1.0 - ((2.0 * inter + eps) / (denom + eps))).mean()


def bce_dice_loss(
    logits: Tensor,
    targets: Tensor,
    pixel_weight: Optional[Tensor] = None,
    pos_weight: float = 2.0, 
) -> Tensor:
    _check_binary_tensors(logits, targets)
 
    pw = torch.tensor([pos_weight], device=logits.device)
    bce = F.binary_cross_entropy_with_logits(
        logits, targets,
        pos_weight=pw,
        reduction='none',
    )
    if pixel_weight is not None:
        bce = (bce * pixel_weight).sum() / pixel_weight.sum().clamp_min(1.0)
    else:
        bce = bce.mean()
 
    return bce + soft_dice_loss(logits, targets)


@torch.no_grad()
def binary_iou_score(
    logits: Tensor, targets: Tensor,
    threshold: float = 0.5, eps: float = 1e-6,
) -> Tensor:
    _check_binary_tensors(logits, targets)
    preds = (torch.sigmoid(logits) >= threshold).to(targets.dtype)
    dims = (1, 2, 3)
    inter = torch.sum(preds * targets, dim=dims)
    union = torch.sum((preds + targets) > 0, dim=dims).to(targets.dtype)
    return ((inter + eps) / (union + eps)).mean()


@torch.no_grad()
def evaluate_iou(
    model: DHFLMedSAM,
    loader: DataLoader,
    device: torch.device,
    max_batches: int = 20,
) -> float:
    model.eval()
    scores = []
    for i, (image, mask) in enumerate(loader):
        if i >= max_batches:
            break
        logits = model(image.to(device))
        scores.append(binary_iou_score(logits, mask.to(device)))
    return float(torch.stack(scores).mean().cpu())


def error_map_from_prediction(
    pred_logits: Tensor, corrected_mask: Tensor
) -> Tensor:
    return corrected_mask.float() - torch.sigmoid(pred_logits).detach()


def error_weight_map(error_map: Tensor, base: float = 1.0, gain: float = 4.0) -> Tensor:
    return base + gain * error_map.abs().detach()


@dataclass(frozen=True)
class FeedbackStepResult:
    loss: float
    iou_before: float
    iou_after: float
    mean_abs_error: float
    delta_abs_before: float
    delta_abs_after: float

## Anchor Regularization

In [11]:
def feedback_step(
    model: DHFLMedSAM,
    optimizer: torch.optim.Optimizer,
    image: Tensor,
    corrected_mask: Tensor,
    anchor: dict,
    error_gain: float = 4.0,
    use_error_weights: bool = True,
    anchor_strength: float = 1e-2,
    correction_denom: float = 0.3,       
    max_grad_norm: float = 1.0,          # clips gradients for MedSAM-scale features
) -> FeedbackStepResult:
    """
    Gradient behaviour:
      Large E  →  correction_signal ≈ 1, seg_loss dominates  →  adapter updates
      Small E  →  correction_signal decays with correction_denom  →  anchor pulls back
      E = 0    →  correction_signal = 0, only anchor remains  →  weights held
 
    correction_denom:
      0.05 (old) — signal stays near 1 unless E is tiny. Too aggressive for BUSI
                   where base IoU~0.50 means mean|E|~0.3 almost always.
      0.30 (new) — signal decays meaningfully as adapter improves on BUSI.
    """
    model.train()
    corrected_mask = corrected_mask.float()
 
    with torch.no_grad():
        before           = model(image, return_debug=True)
        error_map        = error_map_from_prediction(before.logits, corrected_mask)
        iou_before       = binary_iou_score(before.logits, corrected_mask)
        delta_abs_before = before.delta_features.abs().mean()
        mean_abs_error   = error_map.abs().mean()
        correction_signal = mean_abs_error / (mean_abs_error + correction_denom)
 
    pixel_weight = error_weight_map(error_map, gain=error_gain) if use_error_weights else None
 
    optimizer.zero_grad(set_to_none=True)
    out             = model(image, error_map=error_map, return_debug=True)
    seg_loss        = bce_dice_loss(out.logits, corrected_mask, pixel_weight=pixel_weight)
    delta_reg       = 1e-4 * out.delta_features.pow(2).mean()
    correction_loss = (seg_loss + delta_reg) * correction_signal
    anchor_loss     = anchor_regularization_loss(model, anchor, anchor_strength)
    loss            = correction_loss + anchor_loss
    loss.backward()

    torch.nn.utils.clip_grad_norm_(model.adapter_parameters(), max_norm=max_grad_norm)
 
    optimizer.step()
 
    with torch.no_grad():
        after           = model(image, return_debug=True)
        iou_after       = binary_iou_score(after.logits, corrected_mask)
        delta_abs_after = after.delta_features.abs().mean()
 
    return FeedbackStepResult(
        loss=float(loss.detach().cpu()),
        iou_before=float(iou_before.cpu()),
        iou_after=float(iou_after.cpu()),
        mean_abs_error=float(mean_abs_error.cpu()),
        delta_abs_before=float(delta_abs_before.cpu()),
        delta_abs_after=float(delta_abs_after.cpu()),
    )

In [ ]:
def prepare_adapter_only_training(model: DHFLMedSAM) -> dict:
    """Freeze backbone + decoder. Snapshot adapter weights as anchor.
 
    Returns
    -------
    anchor : dict[str, Tensor]
        Detached copy of adapter weights at the start of feedback.
        Pass this to every feedback_step call.
    """
    model.freeze_for_feedback()
    return {
        name: param.detach().clone()
        for name, param in model.adapter.named_parameters()
    }
 
 
def anchor_regularization_loss(
    model: DHFLMedSAM,
    anchor: dict,
    anchor_strength: float,
) -> Tensor:
    """L_anchor = anchor_strength * Σ ||w − w_anchor||²
 
    Penalises drift from the anchor weights.
    When correction_signal → 0, this term dominates and pulls
    the adapter back toward its best-seen checkpoint.
    """
    device = next(model.adapter.parameters()).device
    loss = torch.tensor(0.0, device=device)
    for name, param in model.adapter.named_parameters():
        if name in anchor:
            loss = loss + (param - anchor[name]).pow(2).sum()
    return anchor_strength * loss
 
 
def maybe_update_anchor(
    anchor: dict,
    model: DHFLMedSAM,
    current_iou: float,
    best_iou: float,
    threshold: float = 0.001,
) -> float:
    """Advance the anchor to current weights if IoU is still improving.
 
    While the model keeps improving, the anchor chases it upward 
    on which the adapter is free to keep learning.
 
    Once improvement stalls, the anchor holds at best-seen weights and
    any further drift is penalised, producing a stable plateau.
 
    Returns updated best_iou.
    """
    if current_iou - best_iou > threshold:
        anchor.update({
            name: param.detach().clone()
            for name, param in model.adapter.named_parameters()
        })
        return current_iou
    return best_iou

## Dataset

In [ ]:
class BUSIDataset(Dataset):
    """
    BUSI dataset with 3-way split and optional augmentation for training.
 
    split : 'train' (70%) | 'feedback' (15%) | 'test' (15%)
    augment : True only for split='train' — applied automatically
    """
 
    SPLITS = {
        'train':    (0.00, 0.80),
        'feedback': (0.80, 0.90),
        'test':     (0.90, 1.00),
    }

 
    def __init__(
        self,
        root: str,
        split: str = 'train',
        image_size: int = 256,
        seed: int = 0,
        categories: list = None,
        augment: bool = None,  
    ) -> None:
        assert split in self.SPLITS, f"split must be one of {list(self.SPLITS.keys())}"
        self.image_size = image_size
        self.augment = augment if augment is not None else (split == 'train')
        cats = categories or ['benign', 'malignant']
        pairs = []
 
        for cat in cats:
            cat_dir = os.path.join(root, cat)
            if not os.path.isdir(cat_dir):
                print(f'  [warning] directory not found: {cat_dir}')
                continue
            imgs = sorted([
                f for f in os.listdir(cat_dir)
                if f.endswith('.png') and '_mask' not in f
            ])
            for img_name in imgs:
                stem = img_name.replace('.png', '')
                mask_candidates = sorted([
                    f for f in os.listdir(cat_dir)
                    if f.startswith(stem + '_mask') and f.endswith('.png')
                ])
                if mask_candidates:
                    pairs.append((
                        os.path.join(cat_dir, img_name),
                        [os.path.join(cat_dir, m) for m in mask_candidates],
                    ))
 
        if not pairs:
            raise RuntimeError(f'No image-mask pairs found in {root}.')
 
        rng = torch.Generator().manual_seed(seed)
        idx = torch.randperm(len(pairs), generator=rng).tolist()
        lo, hi = self.SPLITS[split]
        n = len(pairs)
        self.pairs = [pairs[idx[i]] for i in range(int(lo * n), int(hi * n))]
 
        print(f'BUSIDataset [{split:8s}]: {len(self.pairs):3d} samples  '
              f'(augment={self.augment}, categories={cats})')
 
    def __len__(self) -> int:
        return len(self.pairs)
 
    def _augment(self, img: PILImage.Image, msk: PILImage.Image, seed: int):
        """Apply identical geometric transforms to image and mask."""
        rng = random.Random(seed)
 
        # Horizontal flip
        if rng.random() < 0.5:
            img = img.transpose(PILImage.FLIP_LEFT_RIGHT)
            msk = msk.transpose(PILImage.FLIP_LEFT_RIGHT)
 
        # Rotation ±15°
        angle = rng.uniform(-15, 15)
        img = img.rotate(angle, resample=PILImage.BILINEAR,  fillcolor=0)
        msk = msk.rotate(angle, resample=PILImage.NEAREST, fillcolor=0)
 
        # Color jitter on image only (not mask)
        brightness = rng.uniform(0.8, 1.2)
        contrast   = rng.uniform(0.8, 1.2)
        img_arr = np.array(img).astype(np.float32)
        img_arr = img_arr * brightness
        mean    = img_arr.mean()
        img_arr = (img_arr - mean) * contrast + mean
        img_arr = np.clip(img_arr, 0, 255).astype(np.uint8)
        img = PILImage.fromarray(img_arr)
 
        return img, msk
 
    def __getitem__(self, index: int):
        img_path, mask_paths = self.pairs[index]
 
        img = PILImage.open(img_path).convert('RGB').resize(
            (self.image_size, self.image_size), PILImage.BILINEAR
        )
 
        merged = np.zeros((self.image_size, self.image_size), dtype=np.float32)
        for mp in mask_paths:
            m = PILImage.open(mp).convert('L').resize(
                (self.image_size, self.image_size), PILImage.NEAREST
            )
            merged = np.maximum(merged, np.array(m).astype(np.float32) / 255.0)
        msk = PILImage.fromarray((merged * 255).astype(np.uint8), mode='L')
 
        if self.augment:
            img, msk = self._augment(img, msk, seed=index)
 
        img_t = torch.from_numpy(np.array(img)).permute(2, 0, 1).float() / 255.0
        msk_t = (torch.from_numpy(np.array(msk)).float() / 255.0 > 0.5).float().unsqueeze(0)
        return img_t, msk_t
 
 
# Sanity check
print('Dataset split sizes (with augmentation on train):')
_t  = BUSIDataset(DATASET_ROOT, split='train',    image_size=256, seed=0)
_f  = BUSIDataset(DATASET_ROOT, split='feedback', image_size=256, seed=0)
_te = BUSIDataset(DATASET_ROOT, split='test',     image_size=256, seed=0)
_img, _msk = _t[0]
print(f'Sample — image: {tuple(_img.shape)}  mask: {tuple(_msk.shape)}')

Dataset split sizes (with augmentation on train):
BUSIDataset [train   ]: 517 samples  (augment=True, categories=['benign', 'malignant'])
BUSIDataset [feedback]:  65 samples  (augment=False, categories=['benign', 'malignant'])
BUSIDataset [test    ]:  65 samples  (augment=False, categories=['benign', 'malignant'])
Sample — image: (3, 256, 256)  mask: (1, 256, 256)


## Smoke Test

In [28]:
def run_smoke_tests(backbone: nn.Module, device: torch.device) -> None:
    """Tests both decoders using a real BUSI sample."""
    x = torch.randn(2, 3, 256, 256, device=device)
 
    # Use one real BUSI sample for the feedback step test
    _smoke_ds = BUSIDataset(DATASET_ROOT, split='train', image_size=256, seed=42)
    image, mask = _smoke_ds[0]
    image = image.unsqueeze(0).to(device)
    mask  = mask.unsqueeze(0).to(device)
 
    for dtype in ('lightweight', 'heavy'):
        print(f'\n── decoder={dtype} ──')
        model = DHFLMedSAM(backbone, adapter_bottleneck=64, decoder_type=dtype).to(device)
        model.eval()
 
        with torch.no_grad():
            debug = model(x, return_debug=True)
 
        # Output must match INPUT resolution (256×256), not SAM internal (1024×1024)
        assert debug.logits.shape == (2, 1, 256, 256), \
            f'Bad output shape: {debug.logits.shape}'
        assert debug.features.shape == (2, 256, 64, 64), \
            f'Bad feature shape: {debug.features.shape}'
        assert debug.delta_features.shape == debug.features.shape
 
        print(f'  input:    {tuple(x.shape)}')
        print(f'  features: {tuple(debug.features.shape)}  (SAM stride-16 of 1024)')
        print(f'  output:   {tuple(debug.logits.shape)}  ✓ matches input size')
        print(f'  |delta|:  {float(debug.delta_features.abs().mean()):.6f}  (0 expected — zero_init)')
 
        # One feedback step
        model.unfreeze_decoder()
        anchor    = prepare_adapter_only_training(model)
        optimizer = torch.optim.Adam(model.adapter_parameters(), lr=1e-4)
        result    = feedback_step(model, optimizer, image, mask, anchor=anchor)
 
        assert result.loss >= 0.0
        assert all(not p.requires_grad for p in model.backbone.parameters())
        assert all(not p.requires_grad for p in model.decoder.parameters())
        assert all(p.requires_grad     for p in model.adapter.parameters())
 
        with torch.no_grad():
            after = model(image, return_debug=True)
 
        print(f'  iou:      {result.iou_before:.4f} → {result.iou_after:.4f}')
        print(f'  |delta| after 1 step: {float(after.delta_features.abs().mean()):.6f}  (should be > 0)')
 
    print('\nAll smoke tests passed ✓')
 
 
run_smoke_tests(BACKBONE, DEVICE)

BUSIDataset [train   ]: 517 samples  (augment=True, categories=['benign', 'malignant'])

── decoder=lightweight ──
DHFLMedSAM | decoder=lightweight | adapter=74.0K  decoder=588.5K
  input:    (2, 3, 256, 256)
  features: (2, 256, 64, 64)  (SAM stride-16 of 1024)
  output:   (2, 1, 256, 256)  ✓ matches input size
  |delta|:  0.000000  (0 expected — zero_init)
  iou:      0.1145 → 0.1185
  |delta| after 1 step: 0.000296  (should be > 0)

── decoder=heavy ──
DHFLMedSAM | decoder=heavy | adapter=74.0K  decoder=1809.8K
  input:    (2, 3, 256, 256)
  features: (2, 256, 64, 64)  (SAM stride-16 of 1024)
  output:   (2, 1, 256, 256)  ✓ matches input size
  |delta|:  0.000000  (0 expected — zero_init)
  iou:      0.1181 → 0.1187
  |delta| after 1 step: 0.000346  (should be > 0)

All smoke tests passed ✓


## Base Decoder Training

Training only the decoder on top of the frozen MedSAM encoder.
The backbone is already pretrained and in this setup, we are teaching the decoder
to read its features for our specific task.

### TRAINING WITH LIGHTWEIGHT DECODER

In [31]:
def train_decoder(
    model: 'DHFLMedSAM',
    loader: DataLoader,
    steps: int,
    lr: float,
    device: torch.device,
) -> None:
    """
    Train only the decoder (backbone stays frozen).
    Uses CosineAnnealingLR to prevent loss bouncing in late training.
    """
    for p in model.backbone.parameters():
        p.requires_grad_(False)
    model.unfreeze_decoder()
    for p in model.adapter.parameters():
        p.requires_grad_(False)
 
    model.train()
    optimizer = torch.optim.Adam(
        [p for p in model.decoder.parameters() if p.requires_grad], lr=lr
    )
    # Cosine decay: lr → 1e-5 over all steps
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=steps, eta_min=1e-5
    )
 
    batches = cycle(loader)
    for step in range(1, steps + 1):
        image, mask = next(batches)
        image, mask = image.to(device), mask.to(device)
 
        optimizer.zero_grad(set_to_none=True)
        logits = model(image)
        loss = bce_dice_loss(logits, mask)
        loss.backward()
        optimizer.step()
        scheduler.step()   # ← decay LR each step
 
        if step == 1 or step == steps or step % max(steps // 5, 1) == 0:
            current_lr = scheduler.get_last_lr()[0]
            print(f'base step {step:04d}/{steps}  '
                  f'loss={float(loss.detach().cpu()):.4f}  '
                  f'lr={current_lr:.2e}')


# ── RUN BASE TRAINING 

torch.manual_seed(7)
 
IMAGE_SIZE   = 256
BASE_STEPS   = 700
BATCH_SIZE   = 2
DECODER_TYPE = 'lightweight'
 
# Rebuild with new 80/10/10 split
train_ds         = BUSIDataset(DATASET_ROOT, split='train',    image_size=IMAGE_SIZE, seed=7)
feedback_ds      = BUSIDataset(DATASET_ROOT, split='feedback', image_size=IMAGE_SIZE, seed=7)
test_ds          = BUSIDataset(DATASET_ROOT, split='test',     image_size=IMAGE_SIZE, seed=7)
 
train_loader         = DataLoader(train_ds,    batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
feedback_loader      = DataLoader(feedback_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
feedback_eval_loader = DataLoader(feedback_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader          = DataLoader(test_ds,     batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
 
model = DHFLMedSAM(BACKBONE, adapter_bottleneck=64, decoder_type=DECODER_TYPE).to(DEVICE)
 
print(f'\nTraining {DECODER_TYPE} decoder (dropout + augmentation + LR schedule + pos_weight)...')
train_decoder(model, train_loader, steps=BASE_STEPS, lr=1e-3, device=DEVICE)
 
base_state        = copy.deepcopy(model.state_dict())
base_feedback_iou = evaluate_iou(model, feedback_eval_loader, DEVICE)
base_test_iou     = evaluate_iou(model, test_loader, DEVICE)
 
print(f'\nBase model:')
print(f'  feedback IoU: {base_feedback_iou:.4f}')
print(f'  test IoU:     {base_test_iou:.4f}')
print(f'  gap:          {base_feedback_iou - base_test_iou:.4f}  (target: <0.10)')

BUSIDataset [train   ]: 517 samples  (augment=True, categories=['benign', 'malignant'])
BUSIDataset [feedback]:  65 samples  (augment=False, categories=['benign', 'malignant'])
BUSIDataset [test    ]:  65 samples  (augment=False, categories=['benign', 'malignant'])
DHFLMedSAM | decoder=lightweight | adapter=74.0K  decoder=588.5K

Training lightweight decoder (dropout + augmentation + LR schedule + pos_weight)...
base step 0001/700  loss=1.6650  lr=1.00e-03
base step 0140/700  loss=1.2163  lr=9.05e-04
base step 0280/700  loss=1.0383  lr=6.58e-04
base step 0420/700  loss=0.7631  lr=3.52e-04
base step 0560/700  loss=0.7951  lr=1.05e-04
base step 0700/700  loss=0.7709  lr=1.00e-05

Base model:
  feedback IoU: 0.5347
  test IoU:     0.5296
  gap:          0.0052  (target: <0.10)


In [ ]:
CORRECTION_STEPS_LIST = [0, 10, 30, 80, 160, 320]
ANCHOR_STRENGTH       = 1e-2 
FEEDBACK_LR           = 1e-4
 
lightweight_results = []
 
for correction_steps in CORRECTION_STEPS_LIST:
    # Pass decoder_type explicitly — must match base_state
    model = DHFLMedSAM(BACKBONE, adapter_bottleneck=64, decoder_type=DECODER_TYPE).to(DEVICE)
    model.load_state_dict(base_state)
 
    anchor    = prepare_adapter_only_training(model)
    optimizer = torch.optim.Adam(
        model.adapter_parameters(), lr=FEEDBACK_LR, weight_decay=1e-5
    )
    batches  = cycle(train_loader)   # was cycle(feedback_loader) — more variety
    best_iou = base_feedback_iou
 
    for step in range(1, correction_steps + 1):
        image, corrected_mask = next(batches)
        image, corrected_mask = image.to(DEVICE), corrected_mask.to(DEVICE)
 
        results = feedback_step(
            model, optimizer, image, corrected_mask,
            anchor=anchor,
            error_gain=4.0,
            use_error_weights=True,
            anchor_strength=ANCHOR_STRENGTH,
        )
 
        if step % 20 == 0:
            current_iou = evaluate_iou(model, feedback_eval_loader, DEVICE)
            best_iou    = maybe_update_anchor(anchor, model, current_iou, best_iou)
 
        if step in {1, correction_steps} or step % 40 == 0:
            print(
                f'corrections={correction_steps:03d} '
                f'step={step:03d}/{correction_steps} '
                f'loss={results.loss:.4f} '
                f'iou={results.iou_before:.4f}→{results.iou_after:.4f}'
            )
 
    feedback_iou = evaluate_iou(model, feedback_eval_loader, DEVICE)
    test_iou     = evaluate_iou(model, test_loader, DEVICE)
 
    lightweight_results.append({
        'correction_steps':        correction_steps,
        'approx_corrected_images': correction_steps * BATCH_SIZE,
        'feedback_iou':            feedback_iou,
        'test_iou':                test_iou,
        'feedback_delta':          feedback_iou - base_feedback_iou,
        'test_delta':              test_iou - base_test_iou,
    })
 
print('\nCorrection sweep summary (MedSAM + BUSI, adaptive anchor)')
print('steps | approx corrected images | feedback IoU | test IoU | feedback delta | test delta')
for row in lightweight_results:
    print(
        f"{row['correction_steps']:5d} | "
        f"{row['approx_corrected_images']:23d} | "
        f"{row['feedback_iou']:.4f}       | "
        f"{row['test_iou']:.4f}   | "
        f"{row['feedback_delta']:+.4f}         | "
        f"{row['test_delta']:+.4f}"
    )

### TRAINING WITH HEAVY DECODER

In [ ]:
def train_decoder(
    model: 'DHFLMedSAM',
    loader: DataLoader,
    steps: int,
    lr: float,
    device: torch.device,
) -> None:
    """
    Train only the decoder (backbone stays frozen).
    Uses CosineAnnealingLR to prevent loss bouncing in late training.
    """
    for p in model.backbone.parameters():
        p.requires_grad_(False)
    model.unfreeze_decoder()
    for p in model.adapter.parameters():
        p.requires_grad_(False)
 
    model.train()
    optimizer = torch.optim.Adam(
        [p for p in model.decoder.parameters() if p.requires_grad], lr=lr
    )
    # Cosine decay: lr → 1e-5 over all steps
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=steps, eta_min=1e-5
    )
 
    batches = cycle(loader)
    for step in range(1, steps + 1):
        image, mask = next(batches)
        image, mask = image.to(device), mask.to(device)
 
        optimizer.zero_grad(set_to_none=True)
        logits = model(image)
        loss = bce_dice_loss(logits, mask)
        loss.backward()
        optimizer.step()
        scheduler.step()   # ← decay LR each step
 
        if step == 1 or step == steps or step % max(steps // 5, 1) == 0:
            current_lr = scheduler.get_last_lr()[0]
            print(f'base step {step:04d}/{steps}  '
                  f'loss={float(loss.detach().cpu()):.4f}  '
                  f'lr={current_lr:.2e}')


# ── RUN BASE TRAINING 

torch.manual_seed(7)
 
IMAGE_SIZE   = 256
BASE_STEPS   = 700
BATCH_SIZE   = 1
DECODER_TYPE = 'heavy'
 
# Rebuild with new 80/10/10 split
train_ds         = BUSIDataset(DATASET_ROOT, split='train',    image_size=IMAGE_SIZE, seed=7)
feedback_ds      = BUSIDataset(DATASET_ROOT, split='feedback', image_size=IMAGE_SIZE, seed=7)
test_ds          = BUSIDataset(DATASET_ROOT, split='test',     image_size=IMAGE_SIZE, seed=7)
 
train_loader         = DataLoader(train_ds,    batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
feedback_loader      = DataLoader(feedback_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
feedback_eval_loader = DataLoader(feedback_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader          = DataLoader(test_ds,     batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
 
model = DHFLMedSAM(BACKBONE, adapter_bottleneck=64, decoder_type=DECODER_TYPE).to(DEVICE)
 
print(f'\nTraining {DECODER_TYPE} decoder (dropout + augmentation + LR schedule + pos_weight)...')
train_decoder(model, train_loader, steps=BASE_STEPS, lr=1e-3, device=DEVICE)
 
base_state        = copy.deepcopy(model.state_dict())
base_feedback_iou = evaluate_iou(model, feedback_eval_loader, DEVICE)
base_test_iou     = evaluate_iou(model, test_loader, DEVICE)
 
print(f'\nBase model:')
print(f'  feedback IoU: {base_feedback_iou:.4f}')
print(f'  test IoU:     {base_test_iou:.4f}')
print(f'  gap:          {base_feedback_iou - base_test_iou:.4f}  (target: <0.10)')

In [ ]:
CORRECTION_STEPS_LIST = [0, 10, 30, 80, 160, 320]
ANCHOR_STRENGTH       = 1e-2
FEEDBACK_LR           = 1e-4
 
heavy_results = []
 
for correction_steps in CORRECTION_STEPS_LIST:
    # Pass decoder_type explicitly — must match base_state
    model = DHFLMedSAM(BACKBONE, adapter_bottleneck=64, decoder_type=DECODER_TYPE).to(DEVICE)
    model.load_state_dict(base_state)
 
    anchor    = prepare_adapter_only_training(model)
    optimizer = torch.optim.Adam(
        model.adapter_parameters(), lr=FEEDBACK_LR, weight_decay=1e-5
    )
    batches  = cycle(train_loader)   # was cycle(feedback_loader) — more variety
    best_iou = base_feedback_iou
 
    for step in range(1, correction_steps + 1):
        image, corrected_mask = next(batches)
        image, corrected_mask = image.to(DEVICE), corrected_mask.to(DEVICE)
 
        results = feedback_step(
            model, optimizer, image, corrected_mask,
            anchor=anchor,
            error_gain=4.0,
            use_error_weights=True,
            anchor_strength=ANCHOR_STRENGTH,
        )
 
        if step % 20 == 0:
            current_iou = evaluate_iou(model, feedback_eval_loader, DEVICE)
            best_iou    = maybe_update_anchor(anchor, model, current_iou, best_iou)
 
        if step in {1, correction_steps} or step % 40 == 0:
            print(
                f'corrections={correction_steps:03d} '
                f'step={step:03d}/{correction_steps} '
                f'loss={result.loss:.4f} '
                f'iou={result.iou_before:.4f}→{result.iou_after:.4f}'
            )
 
    feedback_iou = evaluate_iou(model, feedback_eval_loader, DEVICE)
    test_iou     = evaluate_iou(model, test_loader, DEVICE)
 
    heavy_results.append({
        'correction_steps':        correction_steps,
        'approx_corrected_images': correction_steps * BATCH_SIZE,
        'feedback_iou':            feedback_iou,
        'test_iou':                test_iou,
        'feedback_delta':          feedback_iou - base_feedback_iou,
        'test_delta':              test_iou - base_test_iou,
    })
 
print('\nCorrection sweep summary (MedSAM + BUSI, adaptive anchor)')
print('steps | approx corrected images | feedback IoU | test IoU | feedback delta | test delta')
for row in heavy_results:
    print(
        f"{row['correction_steps']:5d} | "
        f"{row['approx_corrected_images']:23d} | "
        f"{row['feedback_iou']:.4f}       | "
        f"{row['test_iou']:.4f}   | "
        f"{row['feedback_delta']:+.4f}         | "
        f"{row['test_delta']:+.4f}"
    )

## Notes on Memory & Speed

| Backbone | VRAM needed | Speed (256px, batch=4) | Recommendation |
|---|---|---|---|
| MedSAM ViT-B | ~6 GB | ~2–4 it/s (GPU) | Best quality, needs GPU |
| SAM ViT-B | ~6 GB | ~2–4 it/s (GPU) | Same size as MedSAM |
| ViT-Small (fallback) | ~2 GB | ~8 it/s (GPU) | Fast, weaker features |
| Any of the above | CPU only | Very slow | Fine for debugging only |

**If you get OOM errors:**
- Reduce `BATCH_SIZE` to 2 or 1
- Reduce `IMAGE_SIZE` to 128 (features become [B, 256, 8, 8])
- Use the ViT-Small fallback
- Use `torch.cuda.empty_cache()` between sweep runs

Backbone is always in eval mode and frozen which when the forward pass through it
with no gradient, so the memory cost is lower than full fine-tuning.